# Grid Independence Test

A benchmarking workflow to verify that CFD results are **mesh-independent**.

The experiment fixes **one terrain** and **one wind direction** and runs the same
simulation with a family of progressively finer mesh configurations. If the
flow statistics (e.g. velocity profiles, turbulence quantities) converge as the
mesh is refined, the coarsest mesh that achieves the desired accuracy level
can be selected for the full dataset generation run.

**Workflow overview:**
1. For each mesh variant, copy the fixed terrain inputs and write a modified
   `terrain_config.yaml` with the variant-specific grid / mesh overrides.
2. Run the terrain mesh pipeline (or skip if already done) to produce OpenFOAM
   case inputs — one per variant.
3. Generate OpenFOAM cases and mesh them locally (or on HPC).
4. Submit all variants to SLURM and monitor their progress.
5. Compare key metrics across variants to establish mesh independence.

**Resume-safe:** Close and reopen at any time.  
All decisions are derived from `benchmark_metadata.json` files written into each
variant directory.

## 1. Configuration

Edit these settings before running the notebook.

In [ ]:
import os
import sys

# ── Paths ────────────────────────────────────────────────────────────────────
# Root of the CFD-dataset repository (directory containing this notebook)
REPO_ROOT = os.path.dirname(os.path.abspath("__file__"))

# Path to the single terrain directory to use for this benchmark
# (e.g. a folder produced by generateInputs.py under Data/downloads/)
FIXED_TERRAIN_DIR = os.path.join(REPO_ROOT, "Data", "downloads", "terrain_0001_123_456")

# Single wind direction to test (degrees, 0 = North)
FIXED_ROTATION_DEG = 0

# Base terrain configuration file — variant overrides are layered on top of this
TERRAIN_CONFIG_PATH = os.path.join(REPO_ROOT, "configs", "terrain_config.yaml")

# ABL BC configuration file
ABL_BC_CONFIG_PATH = os.path.join(REPO_ROOT, "configs", "abl_bc_config.yaml")

# Root output directory: one sub-folder will be created per mesh variant
CASES_OUTPUT_DIR = os.path.join(REPO_ROOT, "grid_independence")

# OpenFOAM template and taskmanager config paths
TEMPLATE_PATH = os.path.join(REPO_ROOT, "configs", "openfoam_template")
TASKMANAGER_CONFIG_PATH = os.path.join(REPO_ROOT, "configs", "taskmanager_config.yaml")

# Remote HPC path on Deucalion (used by taskManager for rsync/sbatch)
CLUSTER_PATH = "path/to/remote/directory"

# Number of parallel workers for local meshing
N_PARALLEL_WORKERS = 2

# ── Mesh variants ─────────────────────────────────────────────────────────────
# Each entry must have a unique "name" key.  All other keys are deep-merged
# into the base terrain_config.yaml before running the mesh pipeline.
# Use the same top-level section names as terrain_config.yaml
# ("grid", "mesh", "terrain", "boundary", …).
MESH_VARIANTS = [

    # Example extra variants:
    {
        "name": "fine", 
        "grid": {"nx": 433, "ny": 433},
        "mesh": {"total_z_cells": 90},
    },
    {
        "name": "medium", 
        "grid": {"nx": 309, "ny": 309},
        "mesh": {"total_z_cells": 72},
    },
    {
        "name": "coarse", 
        "grid": {"nx": 271, "ny": 271},
        "mesh": {"total_z_cells": 50},
    }
]

# Validate mesh variants early with a clear error message
for _idx, _variant in enumerate(MESH_VARIANTS):
    if not isinstance(_variant, dict):
        raise TypeError(f"MESH_VARIANTS[{_idx}] must be a dict, got {type(_variant).__name__}")
    if "name" not in _variant:
        raise KeyError(f"MESH_VARIANTS[{_idx}] is missing required key: 'name'")

    _mesh = _variant.get("mesh", {})
    if "z_grading" in _mesh:
        _zg = _mesh["z_grading"]
        _valid_shape = (
            isinstance(_zg, list)
            and all(
                isinstance(_block, (list, tuple)) and len(_block) == 3
                for _block in _zg
            )
        )
        if not _valid_shape:
            raise ValueError(
                f"MESH_VARIANTS[{_idx}]['mesh']['z_grading'] must be like [[len_frac, cell_frac, expansion]]"
            )

# ── SLURM settings ────────────────────────────────────────────────────────────
# These are applied to every submitted job (all variants use the same resources)
SLURM_PARTITION        = "normal-x86" # partition/queue name
SLURM_TIME             = "10:00:00"   # wall-clock time limit per job
SLURM_N_NODES          = 1            # number of nodes to use per job
SLURM_NTASKS_PER_NODE  = 128        # number of tasks (processes) to run in parallel on each node


print(f"REPO_ROOT           : {REPO_ROOT}")
print(f"FIXED_TERRAIN_DIR   : {FIXED_TERRAIN_DIR}")
print(f"FIXED_ROTATION_DEG  : {FIXED_ROTATION_DEG}")
print(f"TERRAIN_CONFIG_PATH : {TERRAIN_CONFIG_PATH}")
print(f"ABL_BC_CONFIG_PATH  : {ABL_BC_CONFIG_PATH}")
print(f"CASES_OUTPUT_DIR    : {CASES_OUTPUT_DIR}")
print(f"N_PARALLEL_WORKERS  : {N_PARALLEL_WORKERS}")
print(f"Mesh variants       : {[v['name'] for v in MESH_VARIANTS]}")

## 2. Imports

In [ ]:
import copy
import json
from dataclasses import asdict
from pathlib import Path
from datetime import datetime
import importlib
import sys

import yaml
import pandas as pd

# ── terrain_mesh (installed package) ────────────────────────────────────────────
try:
    import terrain_mesh as tm
    _MESH_OK = True
    print("✓ terrain_mesh imported")
except ImportError as _e:
    _MESH_OK = False
    print(f"✗ terrain_mesh not available: {_e}")
    print("  Install the required packages: pip install git+https://github.com/souravsud/terrain_following_mesh_generator.git")


def _deep_update(base: dict, updates: dict) -> dict:
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            _deep_update(base[key], value)
        else:
            base[key] = value
    return base

# ── abl_bc_generator (installed package) ────────────────────────────────────────
try:
    from abl_bc_generator import (
        generate_inlet_data_workflow,
        ABLConfig,
        AtmosphericConfig,
        TurbulenceConfig,
        MeshConfig,
        OpenFOAMConfig,
    )

    def load_abl_config(yaml_path: str) -> ABLConfig:
        cfg_dict = asdict(ABLConfig())
        with open(yaml_path, "r", encoding="utf-8") as fh:
            file_cfg = yaml.safe_load(fh) or {}
        if not isinstance(file_cfg, dict):
            raise ValueError(f"Expected YAML mapping at root: {yaml_path}")
        cfg_dict = _deep_update(cfg_dict, file_cfg)
        return ABLConfig(
            atmospheric=AtmosphericConfig(**cfg_dict.get("atmospheric", {})),
            turbulence=TurbulenceConfig(**cfg_dict.get("turbulence", {})),
            mesh=MeshConfig(**cfg_dict.get("mesh", {})),
            openfoam=OpenFOAMConfig(**cfg_dict.get("openfoam", {})),
        )

    _ABL_OK = True
    print("✓ abl_bc_generator imported")
except ImportError as _e:
    _ABL_OK = False
    generate_inlet_data_workflow = None
    ABLConfig = None
    AtmosphericConfig = None
    TurbulenceConfig = None
    MeshConfig = None
    OpenFOAMConfig = None
    print(f"✗ abl_bc_generator not available: {_e}")
    print("  Install the required packages: pip install git+https://github.com/souravsud/ABL_BC_generator.git")

# ── taskManager (submodule) ───────────────────────────────────────────────────
try:
    import taskmanager
    # Force reload to pick up latest changes
    importlib.reload(taskmanager)
    from taskmanager import OpenFOAMCaseGenerator
    _TM_OK = True
    print("✓ taskmanager imported (with reload)")
except ImportError as _e:
    _TM_OK = False
    OpenFOAMCaseGenerator = None
    print(f"✗ taskmanager not available: {_e}")
    print("  Install the required packages: pip install git+https://github.com/souravsud/taskManager.git")

## 3. Generate Mesh Variant Inputs

For each entry in `MESH_VARIANTS`:
1. Create a variant output directory (`grid_bench_{name}_{rotation:03d}deg/`).
2. Load `terrain_config.yaml` and deep-merge the variant overrides.
3. Save the modified config to a `variant_terrain_config.yaml` inside the variant dir.
4. Run the terrain mesh pipeline with the modified config.
5. Generate ABL boundary conditions for the fixed wind direction.
6. Write a `benchmark_metadata.json` to record variant parameters and pipeline status.

Variants that already have a `benchmark_metadata.json` with `status == "complete"`
are **skipped** (resume-safe).

In [ ]:
def _deep_merge(base: dict, overrides: dict) -> dict:
    """Recursively merge *overrides* into a deep copy of *base*."""
    result = copy.deepcopy(base)
    for key, value in overrides.items():
        if key == "name":
            continue  # skip the variant name key
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = _deep_merge(result[key], value)
        else:
            result[key] = copy.deepcopy(value)
    return result


def _variant_dir(cases_output_dir: str, name: str, rotation: int) -> Path:
    return Path(cases_output_dir) / f"grid_bench_{name}_{rotation:03d}deg"


def _mesh_params_summary(variant: dict) -> str:
    """One-liner summary of the mesh parameters for display in tables."""
    parts = []
    if "grid" in variant:
        g = variant["grid"]
        if "nx" in g and "ny" in g:
            parts.append(f"nx={g['nx']}, ny={g['ny']}")
    if "mesh" in variant:
        m = variant["mesh"]
        if "total_z_cells" in m:
            parts.append(f"nz={m['total_z_cells']}")
    return "; ".join(parts) if parts else str(variant)


# ── Load base config ──────────────────────────────────────────────────────────
with open(TERRAIN_CONFIG_PATH) as fh:
    base_config = yaml.safe_load(fh)
print(f"✓ Loaded base config: {TERRAIN_CONFIG_PATH}")

if _ABL_OK:
    abl_base_config = load_abl_config(ABL_BC_CONFIG_PATH)
    print(f"✓ Loaded ABL BC config: {ABL_BC_CONFIG_PATH}")
else:
    abl_base_config = None

# ── Locate terrain inputs (DEM / roughness) ───────────────────────────────────
terrain_path = Path(FIXED_TERRAIN_DIR)
if not terrain_path.exists():
    print(f"✗ FIXED_TERRAIN_DIR not found: {FIXED_TERRAIN_DIR}")
    print("  Set FIXED_TERRAIN_DIR to an existing terrain directory and re-run.")
else:
    dem_files  = sorted([f for f in terrain_path.glob("terrain_*.tif") if "_raw" not in f.name])
    rmap_files = sorted([f for f in terrain_path.glob("roughness_*.tif") if "_raw" not in f.name])
    dem_file       = str(dem_files[0])  if dem_files  else None
    roughness_file = str(rmap_files[0]) if rmap_files else None
    print(f"✓ Terrain dir    : {terrain_path.name}")
    print(f"  DEM file       : {Path(dem_file).name if dem_file else 'NOT FOUND'}")
    print(f"  Roughness file : {Path(roughness_file).name if roughness_file else 'not found'}")

# ── Generate variant inputs ───────────────────────────────────────────────────
Path(CASES_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

for variant in MESH_VARIANTS:
    name       = variant["name"]
    var_dir    = _variant_dir(CASES_OUTPUT_DIR, name, FIXED_ROTATION_DEG)
    meta_file  = var_dir / "benchmark_metadata.json"

    # ── Resume check ─────────────────────────────────────────────────────────
    if meta_file.exists():
        with open(meta_file) as fh:
            meta = json.load(fh)
        if meta.get("status") == "complete":
            print(f"  ↷ {name}: already complete, skipping")
            continue

    var_dir.mkdir(parents=True, exist_ok=True)

    # ── Build modified config ─────────────────────────────────────────────────
    variant_config = _deep_merge(base_config, variant)
    variant_config.setdefault("terrain", {})
    variant_config["terrain"]["rotation_deg"] = FIXED_ROTATION_DEG

    variant_config_path = var_dir / "variant_terrain_config.yaml"
    with open(variant_config_path, "w") as fh:
        yaml.dump(variant_config, fh, default_flow_style=False, sort_keys=False)

    # ── Initial metadata ──────────────────────────────────────────────────────
    meta = {
        "name"        : name,
        "mesh_params" : _mesh_params_summary(variant),
        "terrain_dir" : str(FIXED_TERRAIN_DIR),
        "rotation"    : FIXED_ROTATION_DEG,
        "status"      : "pending",
        "created_at"  : datetime.now().isoformat(),
    }
    with open(meta_file, "w") as fh:
        json.dump(meta, fh, indent=2)

    # ── Run mesh pipeline ─────────────────────────────────────────────────────
    if not _MESH_OK:
        print(f"  ○ {name}: mesh pipeline not available (submodule missing)")
        continue

    if not terrain_path.exists() or dem_file is None:
        print(f"  ✗ {name}: FIXED_TERRAIN_DIR missing or has no DEM — skipping mesh step")
        continue

    print(f"  ▶ {name}: running mesh pipeline…")
    try:
        mesh_config = tm.load_config(str(variant_config_path))
        pipeline    = tm.TerrainMeshPipeline()
        pipeline.run(
            dem_path=dem_file,
            rmap_path=roughness_file,
            output_dir=str(var_dir),
            **mesh_config,
        )

        # ── Run ABL BC generation ─────────────────────────────────────────
        if _ABL_OK:
            inletBC_config = copy.deepcopy(abl_base_config)
            inletBC_config.flow_dir_deg = FIXED_ROTATION_DEG
            generate_inlet_data_workflow(str(var_dir), inletBC_config)
            print(f"  ✓ {name}: ABL BC generation complete")
        else:
            print(f"  ⚠ {name}: abl_bc_generator not available — BCs not generated")

        meta["status"] = "complete"
        print(f"  ✓ {name}: mesh pipeline complete")
    except Exception as exc:
        meta["status"] = "failed"
        meta["error"]  = str(exc)
        print(f"  ✗ {name}: {exc}")

    meta["updated_at"] = datetime.now().isoformat()
    with open(meta_file, "w") as fh:
        json.dump(meta, fh, indent=2)

print("\nVariant input generation complete.")

## 4. Status Scanner

Reads every `benchmark_metadata.json` found under `CASES_OUTPUT_DIR` and assembles a
pandas DataFrame.  **Re-run this cell at any time to refresh the view.**

In [ ]:
def scan_benchmark_status(cases_output_dir: str) -> pd.DataFrame:
    """Scan variant directories and return a status DataFrame."""
    output_path = Path(cases_output_dir)
    records = []

    if not output_path.exists():
        print(f"⚠  Output directory does not exist yet: {cases_output_dir}")
        print("   Run Section 3 first to generate variant inputs.")
    else:
        for meta_file in sorted(output_path.glob("grid_bench_*/benchmark_metadata.json")):
            try:
                with open(meta_file) as fh:
                    meta = json.load(fh)
            except (json.JSONDecodeError, OSError) as exc:
                print(f"⚠  Could not read {meta_file}: {exc}")
                continue

            # Check for a sibling case_status.json written by taskmanager
            case_status_file = meta_file.parent / "case_status.json"
            case_status      = {}
            if case_status_file.exists():
                try:
                    with open(case_status_file) as fh:
                        case_status = json.load(fh)
                except (json.JSONDecodeError, OSError):
                    pass

            records.append({
                "variant_name"    : meta.get("name"),
                "mesh_params"     : meta.get("mesh_params"),
                "pipeline_status" : meta.get("status"),
                "mesh_status"     : case_status.get("mesh_status"),
                "job_id"          : case_status.get("job_id"),
                "job_status"      : case_status.get("job_status"),
                "last_checked"    : case_status.get("last_checked"),
                "variant_dir"     : str(meta_file.parent),
            })

    columns = [
        "variant_name", "mesh_params", "pipeline_status",
        "mesh_status", "job_id", "job_status", "last_checked", "variant_dir",
    ]
    return pd.DataFrame(records, columns=columns) if records else pd.DataFrame(columns=columns)


df_bench = scan_benchmark_status(CASES_OUTPUT_DIR)
print(f"Scanned {len(df_bench)} variant(s) at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 5. Summary Dashboard

In [ ]:
STATUS_ICONS = {
    "complete"      : "★",
    "pending"       : "○",
    "failed"        : "✗",
    "ready_to_mesh" : "○",
    "meshing"       : "⟳",
    "meshed"        : "✓",
    "running"       : "▶",
}

total = len(df_bench)
print(f"{'='*55}")
print(f"  GRID INDEPENDENCE TEST — STATUS SUMMARY")
print(f"  Terrain  : {Path(FIXED_TERRAIN_DIR).name}")
print(f"  Rotation : {FIXED_ROTATION_DEG}°")
print(f"  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*55}")
print(f"  Total variants : {total}")
print()
if total > 0:
    counts = df_bench["pipeline_status"].value_counts()
    for status, n in counts.items():
        icon = STATUS_ICONS.get(str(status), "?")
        print(f"  {icon}  {str(status):<16} : {n}")
print(f"{'='*55}")

In [ ]:
# ── Full variant table ────────────────────────────────────────────────────────
if not df_bench.empty:
    display(
        df_bench[
            ["variant_name", "mesh_params", "pipeline_status",
             "mesh_status", "job_id", "job_status", "last_checked"]
        ].reset_index(drop=True)
    )
else:
    print("No variants found. Run Section 3 first.")

## 6. Generate & Mesh Cases

Uses `taskManager` to convert each variant input directory into a full OpenFOAM case
and then mesh it locally using `blockMesh` + `snappyHexMesh`.

**Resume-safe:** variants that already have `mesh_status == DONE` in their
`case_status.json` are skipped.

In [ ]:
if not _TM_OK:
    print("✗ taskmanager not available — cannot generate or mesh cases.")
else:
    from shutil import copytree

    # ── Initialise the case generator ─────────────────────────────────────────
    generator = OpenFOAMCaseGenerator(
        template_path=TEMPLATE_PATH,
        input_dir=CASES_OUTPUT_DIR,
        output_dir=CASES_OUTPUT_DIR,
        cluster_path=CLUSTER_PATH,
        config_path=TASKMANAGER_CONFIG_PATH,
    )
    print(f"✓ OpenFOAMCaseGenerator initialised with config: {generator.config_path}")

    # Re-scan to pick up the latest state
    df_bench = scan_benchmark_status(CASES_OUTPUT_DIR)

    # ── Ensure each completed variant is a runnable OpenFOAM case ─────────────
    for _, row in df_bench.iterrows():
        name = row["variant_name"]

        if row["pipeline_status"] != "complete":
            print(f"  ○ {name}: pipeline_status={row['pipeline_status']}, skipping case preparation")
            continue

        case_dir = Path(row["variant_dir"])

        try:
            # 1) Copy OpenFOAM template files (does not overwrite variant outputs)
            copytree(Path(TEMPLATE_PATH), case_dir, dirs_exist_ok=True)

            # 2) Build render context from pipeline metadata + defaults
            pmeta_path = case_dir / "pipeline_metadata.json"
            with open(pmeta_path) as fh:
                pmeta = json.load(fh)

            bounds = (
                pmeta.get("processing_results", {})
                    .get("grid_statistics", {})
                    .get("bounds", [])
            )
            if len(bounds) >= 4:
                easting = 0.5 * (float(bounds[0]) + float(bounds[1]))
                northing = 0.5 * (float(bounds[2]) + float(bounds[3]))
            else:
                easting, northing = 0.0, 0.0

            cfg = pmeta.get("configurations", {})
            context = {
                "terrain_index": name,
                "rotation_degree": int(cfg.get("terrain", {}).get("rotation_deg", FIXED_ROTATION_DEG)),
                "location": str(pmeta.get("input_files", {}).get("dem_path", "")),
                "end_time": 20000,
                "write_interval": 5000,
                "n_procs": int(SLURM_N_NODES * SLURM_NTASKS_PER_NODE),
                "wind_direction": int(cfg.get("terrain", {}).get("rotation_deg", FIXED_ROTATION_DEG)),
                "easting": easting,
                "northing": northing,
            }

            # 3) Render system dictionaries from .j2 templates
            for rel in ["system/controlDict.j2", "system/decomposeParDict.j2", "system/fvSolution.j2"]:
                j2_file = case_dir / rel
                if j2_file.exists():
                    generator.render_j2_file(j2_file, context)

            # 4) Render HPC script and initialise status tracking
            generator.render_hpc_script(case_dir, case_dir.name)
            generator.initialize_case_status(case_dir)

            print(f"  ✓ {name}: case prepared at {case_dir}")

        except Exception as exc:
            print(f"  ✗ {name}: case preparation failed: {exc}")

    # ── Mesh cases that are ready ──────────────────────────────────────────────
    df_bench = scan_benchmark_status(CASES_OUTPUT_DIR)
    ready_cases = [
        str(Path(row["variant_dir"]))
        for _, row in df_bench.iterrows()
        if row["pipeline_status"] == "complete" and row["mesh_status"] in (None, "", "NOT_RUN", "FAILED", "ERROR")
    ]

    if not ready_cases:
        print("\nNo cases ready for meshing.")
    else:
        print(f"\nMeshing {len(ready_cases)} case(s) with {N_PARALLEL_WORKERS} worker(s)…")
        results = generator.mesh_cases_parallel(ready_cases, n_workers=N_PARALLEL_WORKERS)
        succeeded = sum(results)
        failed    = len(results) - succeeded
        print(f"Meshing complete: {succeeded} succeeded, {failed} failed.")
        print("Re-run Section 4 to refresh the dashboard.")

## 7. Submit to HPC

Submits all meshed (but not yet submitted) variants to SLURM via the taskManager.
All variants use the same SLURM resource settings defined in Section 1.

In [ ]:
if not _TM_OK:
    print("✗ taskmanager not available — cannot submit jobs.")
else:
    import re

    # Ensure generator exists even if Section 6 was not run in this kernel
    if "generator" not in globals():
        generator = OpenFOAMCaseGenerator(
            template_path=TEMPLATE_PATH,
            input_dir=CASES_OUTPUT_DIR,
            output_dir=CASES_OUTPUT_DIR,
            cluster_path=CLUSTER_PATH,
            config_path=TASKMANAGER_CONFIG_PATH,
        )
        print(f"✓ OpenFOAMCaseGenerator initialised with config: {generator.config_path}")

    # Apply Section 1 SLURM resources to this submission run
    generator.hpc_defaults.update({
        "partition": SLURM_PARTITION,
        "walltime": SLURM_TIME,
        "nodes": SLURM_N_NODES,
        "ntasks": int(SLURM_N_NODES * SLURM_NTASKS_PER_NODE),
    })

    def _enforce_slurm_directives(case_path: Path):
        script = case_path / "openfoam.sh"

        if not script.exists():
            j2_script = case_path / "openfoam.sh.j2"
            if j2_script.exists():
                generator.render_hpc_script(case_path, case_path.name)
            else:
                return

        text = script.read_text()
        directives = {
            "partition": SLURM_PARTITION,
            "time": SLURM_TIME,
            "nodes": SLURM_N_NODES,
            "ntasks": int(SLURM_N_NODES * SLURM_NTASKS_PER_NODE),
        }

        for key, value in directives.items():
            pattern = rf"^#SBATCH --{key}=.*$"
            replacement = f"#SBATCH --{key}={value}"
            if re.search(pattern, text, flags=re.MULTILINE):
                text = re.sub(pattern, replacement, text, flags=re.MULTILINE)

        script.write_text(text)

    df_bench = scan_benchmark_status(CASES_OUTPUT_DIR)

    meshed_cases = df_bench[
        (df_bench["mesh_status"] == "DONE") &
        (df_bench["job_id"].isna())
    ]

    if meshed_cases.empty:
        print("No meshed cases ready for submission.")
        print("Possible reasons:")
        print("  • Meshing has not finished yet — re-run Section 6.")
        print("  • All variants have already been submitted.")
    else:
        print(f"Submitting {len(meshed_cases)} variant(s) to SLURM…")
        for _, row in meshed_cases.iterrows():
            name = row["variant_name"]
            case_path = Path(row["variant_dir"])
            try:
                _enforce_slurm_directives(case_path)
                generator.copy_and_submit(case_path)
                status = generator.get_status(case_path) or {}
                job_id = status.get("job_id")

                if status.get("submitted") and job_id:
                    print(f"  ✓ {name}: submitted (job_id={job_id})")
                else:
                    print(f"  ⚠ {name}: no job_id returned; check SSH/SLURM output above")
            except Exception as exc:
                print(f"  ✗ {name}: submission failed: {exc}")

        print("\nSubmission complete. Re-run Section 4 to refresh the dashboard.")

## 8. Refresh Job Statuses

Refreshes SLURM status for submitted variants and writes updates to each `case_status.json`.
If a case is already `job_status == "COMPLETED"` and `results_fetched == True`, it is treated as complete locally and skipped (no SSH/SLURM query).

In [ ]:
if not _TM_OK:
    print("✗ taskmanager not available.")
else:
    import traceback
    
    df_bench = scan_benchmark_status(CASES_OUTPUT_DIR)
    submitted = df_bench[df_bench["job_id"].notna()]

    if submitted.empty:
        print("No submitted jobs to refresh.")
    else:
        refreshed = 0
        skipped_local_complete = 0
        print(f"Refreshing {len(submitted)} job status(es)…")
        
        for _, row in submitted.iterrows():
            case_path = Path(row["variant_dir"])
            try:
                status = generator.get_status(case_path) or {}
                if (
                    str(status.get("job_status", "")).upper() == "COMPLETED"
                    and status.get("results_fetched", False)
                ):
                    skipped_local_complete += 1
                    print(f"  {row['variant_name']}: COMPLETED (results already fetched; skipped remote check)")
                    continue

                new_status = generator.update_job_status(case_path)
                refreshed += 1
                print(f"  {row['variant_name']}: {new_status}")
            except Exception as exc:
                print(f"  ✗ {row['variant_name']}: {exc}")
                print("    Full traceback:")
                traceback.print_exc()

        print("\nJob status refresh complete.")
        print(f"  Refreshed from cluster: {refreshed}")
        print(f"  Skipped as already complete locally: {skipped_local_complete}")
        print("Re-run Section 4 to refresh the dashboard.")

## 9. Fetch Results from HPC

Once jobs have completed (`job_status == "COMPLETED"`), download the results back to the local case 
directories. This includes:
- The **last timestep directory** (e.g., `case/20000/`)
- **postProcessing/** folder (if it exists)
- **Log files** (e.g., `log.simpleFoam`, `log.decomposePar`, etc.)  
  *(excluding blockMesh and checkMesh which are already local)*

This is resume-safe — already fetched cases are skipped.

In [ ]:
if not _TM_OK:
    print("✗ taskmanager not available.")
else:
    # Ensure generator exists even if previous sections were not run
    if "generator" not in globals():
        generator = OpenFOAMCaseGenerator(
            template_path=TEMPLATE_PATH,
            input_dir=CASES_OUTPUT_DIR,
            output_dir=CASES_OUTPUT_DIR,
            cluster_path=CLUSTER_PATH,
            config_path=TASKMANAGER_CONFIG_PATH,
        )
        print(f"✓ OpenFOAMCaseGenerator initialised with config: {generator.config_path}")
    
    # Re-scan to pick up the latest status
    df_bench = scan_benchmark_status(CASES_OUTPUT_DIR)
    
    # Find completed (or failed) jobs that haven't had results fetched yet
    completed_or_failed = df_bench[
        (df_bench["job_status"].isin(["COMPLETED", "FAILED", "TIMEOUT"])) |
        (df_bench["job_status"].str.upper().isin(["COMPLETED", "FAILED", "TIMEOUT"]))
    ]
    
    # Filter out those already fetched (check if case_status has results_fetched)
    cases_to_fetch = []
    for _, row in completed_or_failed.iterrows():
        case_path = Path(row["variant_dir"])
        status = generator.get_status(case_path)
        
        # Only fetch if not already fetched
        if not status or not status.get("results_fetched", False):
            cases_to_fetch.append(case_path)
    
    if not cases_to_fetch:
        print("No completed jobs with unfetched results.")
        print("Either:")
        print("  • No jobs have completed yet — re-run Section 8 to check status")
        print("  • All results have already been fetched")
    else:
        print(f"Fetching results from {len(cases_to_fetch)} variant(s)…\n")
        
        # Fetch results with defaults: last timestep, postProcessing, and log files
        results = generator.fetch_multiple_results(
            cases_to_fetch,
            fetch_last_timestep=True,
            fetch_postprocessing=True,
            fetch_logs=True
        )
        
        # Summary
        succeeded = sum(results)
        failed = len(results) - succeeded
        print(f"\n✓ Results fetch complete: {succeeded} succeeded, {failed} failed.")
        print("Re-run Section 4 to refresh the dashboard to see updated status.")
